# Watch fill updates arrive in Feather

Create a `FeatherSink`, pass it to `Context(sinks=[sink])`, and fill named books normally. A native Tokio worker batches and writes level updates while the Python fill loop runs in a separate thread.

This notebook sends **6,000 deterministic random orders**. It prints committed rows read **from the growing file** before the producer finishes, then reads the completed Feather file with Polars.

Build the bindings with `make py` before running. This example needs no market-data download.

In [ ]:
# ruff: noqa: S101, S311
import asyncio
import random
from pathlib import Path
from time import sleep
from uuid import UUID

import polars as pl

from lobo.context import Context
from lobo.orders import LimitOrder, MarketOrder, Side
from lobo.sinks import FeatherSink, read_feather

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
OUTPUT = ROOT / "notebooks" / "artifacts" / "feather_fills.feather"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ORDER_COUNT = 6_000
TRADER = UUID(int=42)

sink = FeatherSink(
    OUTPUT,
    batch_size=128,
    buffer_size=65_536,
    overwrite=True,  # Explicitly replace this notebook's previous output on rerun.
)
print(sink)

## Tuning and reading while the writer is open

The defaults are **8,192 rows per batch** and a **262,144-byte IO buffer**. Smaller batches expose updates sooner and perform more writes. Here, 128-row batches make progress easy to observe. A partial batch is flushed when the context closes.

Ordinary Feather readers require the closing footer. `read_feather(sink)` reads only the file prefix containing fully written batches and decodes it as Arrow IPC; it never reads an unfinished batch. This inspection copies the committed prefix and belongs outside performance measurements.

The persisted price is a 16-byte big-endian integer. Side is `0 = Buy`, `1 = Sell`. The display helper decodes these values without changing the saved file.

In [ ]:
def readable(frame: pl.DataFrame) -> pl.DataFrame:
    """Make a small selection of persisted updates easy to read."""
    return frame.with_columns(
        pl.col("price").map_elements(
            lambda value: int.from_bytes(value, "big"), return_dtype=pl.UInt64
        ),
        pl.col("side").replace_strict({0: "Buy", 1: "Sell"}),
    ).select(
        "sequence_number",
        "book_id",
        "side",
        "price",
        "visible_quantity",
        "hidden_quantity",
        "number_of_orders",
    )


def fill_random_orders() -> int:
    """Own the context in a background thread and finalize it on exit."""
    rng = random.Random(42)
    with Context(sinks=[sink]) as context:
        book = context.book("RANDOM-FILLS")
        for index in range(ORDER_COUNT):
            side = Side.Buy if rng.randrange(2) == 0 else Side.Sell
            quantity = rng.randint(1, 20)
            if index % 5 == 4:
                order = MarketOrder(quantity, TRADER, side)
            else:
                order = LimitOrder(rng.randint(98, 102), quantity, TRADER, side)
            book.fill(order)
            if (index + 1) % 128 == 0:
                # Demonstration pacing only; the benchmark contains no sleeps.
                sleep(0.02)
    return ORDER_COUNT

## Fill and observe concurrently

The notebook's asyncio task watches the file while `asyncio.to_thread` runs the fill loop. The native sink writes on its own worker. Each printed table contains the latest newly committed updates; a zero order count records a depleted level.

In [ ]:
fill_task = asyncio.create_task(asyncio.to_thread(fill_random_orders))
last_rows = 0
live_snapshots = 0

try:
    while not fill_task.done():
        await asyncio.sleep(0.15)
        if not sink.started or sink.rows_written <= last_rows:
            continue
        snapshot = await asyncio.to_thread(read_feather, sink)
        added = snapshot.slice(last_rows)
        if not fill_task.done():
            live_snapshots += 1
        print(
            f"{snapshot.height:,} committed rows | "
            f"{sink.batches_written} batches | {sink.bytes_written:,} bytes | "
            f"producer running: {not fill_task.done()}"
        )
        print(readable(added.tail(4)))
        last_rows = snapshot.height
finally:
    submitted = await fill_task

assert submitted == ORDER_COUNT
assert live_snapshots > 0, "Expected to observe rows before fills finished"
assert sink.finished
print(f"Submitted {submitted:,} orders; observed {live_snapshots} live snapshots.")

## Read the finished Feather file

Context exit drains the receiver, writes the final partial batch and footer, and closes the writer before the fill task returns. The file now works with normal Feather V2 / Arrow IPC readers.

These are **level-state updates**, not one row per submitted order. An order can fill several makers, and an unmatched market order produces no level mutation.

In [ ]:
updates = pl.read_ipc(OUTPUT)
assert updates.height == sink.rows_written
assert updates["sequence_number"].to_list() == list(range(1, updates.height + 1))
assert updates["book_id"].unique().to_list() == ["RANDOM-FILLS"]
assert updates.equals(read_feather(sink))

print(f"Saved {updates.height:,} updates to {OUTPUT}")
print(f"File size: {OUTPUT.stat().st_size:,} bytes; batches: {sink.batches_written}")
print("First updates:")
print(readable(updates.head(8)))
print("Final updates (including the final partial batch):")
print(readable(updates.tail(8)))

In [ ]:
print(
    readable(updates)
    .group_by("side")
    .agg(
        pl.len().alias("updates"),
        (pl.col("number_of_orders") == 0).sum().alias("depleted_level_updates"),
    )
    .sort("side")
)

## Benchmark the full ITCH file

Run from the repository root:

```bash
make bench-py-feather
make bench-py-record-feather
make bench-py-compare-feather

# Single-threaded replay, with the Feather writer still in the background:
make bench-py-feather-single-threaded

# Full file, AAPL updates only:
make bench-py-feather-aapl
make bench-py-feather-aapl-single-threaded
```

To tune the run, pass `PY_BENCH_OPTIONS`, for example:

```bash
make bench-py-feather PY_BENCH_OPTIONS="--batch-size 8192 --buffer-size 262144 --processes 1 --values 3 --warmups 1 --loops 1"
```

The benchmark replays all tickers through the full ITCH file, with no cutoff, into a Feather sink. It uses `data/NASDAQ/01302020.NASDAQ_ITCH50` by default, like the other replay benchmarks; set `LOBO_ITCH_PATH` to use another file. The timed invocation includes context setup, replay, draining, file finalization, and cleanup. At the end of each timing sample, the benchmark prints rows, finalized file size in bytes/GiB, batches, and the mean setup/replay and remaining drain/finalization times. Printing is excluded from the measurement. Writes overlap replay, so the final drain time does not measure all sink work. Multithreaded and single-threaded targets pass `concurrent=True` and `concurrent=False` respectively, with distinct names and baseline files. There is no default time limit for full-file runs; an explicit `--timeout` can be passed in `PY_BENCH_OPTIONS`. Each invocation uses its own temporary directory and deletes the Feather output after closing, including when an exception occurs or pyperf kills a worker on an explicit timeout. The parent process owns the temporary directory for worker cleanup.

This notebook keeps `notebooks/artifacts/feather_fills.feather` so you can inspect it afterward. Re-running explicitly overwrites only that demo file.